# Financial Engine (Core Logic)

In this notebook, I build the **financial engine** for my South African property investment project.

The purpose of this notebook is to take the expanded **sales dataset** and **rental dataset**, then calculate the core investment metrics that an investor would normally compute manually in a spreadsheet.

I am using the financed-property logic from the attached financial notes, including:
- deposit-based acquisition
- bank loan / bond repayment
- operating expenses
- vacancy and maintenance assumptions
- cash flow
- DSCR
- rental yield
- ROI
- equity growth logic


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the expanded datasets

I load the attached expanded sales and rental CSV files directly.

The financial engine will enrich the sales dataset, because that is the dataset where the investment decision is made.

The rental dataset is used to derive rental benchmarks that can be matched back to each sale listing.

In [2]:
sales_path = Path("data\processed\eda_ready_sales_data.csv")
rentals_path = Path("\data\processed\eda_ready_rentals_data.csv")

df_sales = pd.read_csv(sales_path)
df_rentals = pd.read_csv(rentals_path)

print("Sales shape  :", df_sales.shape)
print("Rentals shape:", df_rentals.shape)

Sales shape  : (9009, 26)
Rentals shape: (10000, 29)


In [3]:
print("Sales columns:")
print(df_sales.columns.tolist())
print()
print("Rental columns:")
print(df_rentals.columns.tolist())

Sales columns:
['source_site', 'listing_id', 'listing_url', 'title', 'purchase_price', 'suburb', 'city', 'province', 'property_type', 'bedrooms', 'bathrooms', 'parking_spaces', 'garage', 'floor_area_sqm', 'land_area_sqm', 'levies', 'rates_taxes', 'description', 'listing_date', 'scraped_timestamp', 'pp_transaction_slug', 'pp_province_slug', 'pp_metro_slug', 'pp_city_slug', 'pp_suburb_slug', 'purchase_price_rands']

Rental columns:
['source_site', 'listing_id', 'rental_url', 'title', 'monthly_rent', 'suburb', 'city', 'province', 'property_type', 'bedrooms', 'bathrooms', 'parking_spaces', 'garage', 'floor_area_sqm', 'land_area_sqm', 'levies', 'rates_taxes', 'furnished_flag', 'availability_text', 'description', 'listing_date', 'scraped_timestamp', 'pp_transaction_slug', 'pp_province_slug', 'pp_metro_slug', 'pp_city_slug', 'pp_suburb_slug', 'log_rent', 'monthly_rent_capped']


## 2. Validate critical column names

Before running any financial logic, I validate that the exact fields I need are present.


In [6]:
required_sales_cols = [
    "listing_id",
    "suburb",
    "city",
    "province",
    "property_type",
    "bedrooms",
    "bathrooms",
    "levies",
    "rates_taxes",
    "purchase_price",
]

required_rental_cols = [
    "listing_id",
    "suburb",
    "city",
    "province",
    "property_type",
    "bedrooms",
    "bathrooms",
    "monthly_rent",
]

missing_sales = [c for c in required_sales_cols if c not in df_sales.columns]
missing_rentals = [c for c in required_rental_cols if c not in df_rentals.columns]

if missing_sales:
    raise ValueError(f"Missing sales columns: {missing_sales}")

if missing_rentals:
    raise ValueError(f"Missing rental columns: {missing_rentals}")

print("All required columns are present.")
print("Confirmed critical sales price column: purchase_price")

All required columns are present.
Confirmed critical sales price column: purchase_price


## 3. Basic type cleanup

I convert the main numeric fields into numeric types so the calculations are stable.


In [7]:
sales_numeric_cols = [
    "purchase_price",
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "garage",
    "floor_area_sqm",
    "land_area_sqm",
    "levies",
    "rates_taxes",
]

rental_numeric_cols = [
    "monthly_rent",
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "garage",
    "floor_area_sqm",
    "land_area_sqm",
    "levies",
    "rates_taxes",
]

for col in sales_numeric_cols:
    if col in df_sales.columns:
        df_sales[col] = pd.to_numeric(df_sales[col], errors="coerce")

for col in rental_numeric_cols:
    if col in df_rentals.columns:
        df_rentals[col] = pd.to_numeric(df_rentals[col], errors="coerce")

df_sales["suburb"] = df_sales["suburb"].astype("string").str.strip()
df_sales["city"] = df_sales["city"].astype("string").str.strip()
df_sales["province"] = df_sales["province"].astype("string").str.strip()
df_sales["property_type"] = df_sales["property_type"].astype("string").str.strip()

df_rentals["suburb"] = df_rentals["suburb"].astype("string").str.strip()
df_rentals["city"] = df_rentals["city"].astype("string").str.strip()
df_rentals["province"] = df_rentals["province"].astype("string").str.strip()
df_rentals["property_type"] = df_rentals["property_type"].astype("string").str.strip()

## 4. Build rental benchmarks

To estimate realistic rent for each sale listing, I create benchmark tables from the rental dataset.

I do this in layers:
1. suburb + property_type + bedrooms
2. suburb + bedrooms
3. suburb only
4. city + property_type + bedrooms
5. city only
6. province only

This fallback structure helps me get an estimated rent even when an exact match is missing.

In [8]:
rent_group_exact = (
    df_rentals.groupby(["suburb", "property_type", "bedrooms"], dropna=False)["monthly_rent"]
    .agg(["median", "mean", "count"])
    .reset_index()
    .rename(columns={
        "median": "rent_median_suburb_type_bed",
        "mean": "rent_mean_suburb_type_bed",
        "count": "rent_count_suburb_type_bed",
    })
)

rent_group_suburb_bed = (
    df_rentals.groupby(["suburb", "bedrooms"], dropna=False)["monthly_rent"]
    .agg(["median", "mean", "count"])
    .reset_index()
    .rename(columns={
        "median": "rent_median_suburb_bed",
        "mean": "rent_mean_suburb_bed",
        "count": "rent_count_suburb_bed",
    })
)

rent_group_suburb = (
    df_rentals.groupby(["suburb"], dropna=False)["monthly_rent"]
    .agg(["median", "mean", "count"])
    .reset_index()
    .rename(columns={
        "median": "rent_median_suburb",
        "mean": "rent_mean_suburb",
        "count": "rent_count_suburb",
    })
)

rent_group_city_type_bed = (
    df_rentals.groupby(["city", "property_type", "bedrooms"], dropna=False)["monthly_rent"]
    .agg(["median", "mean", "count"])
    .reset_index()
    .rename(columns={
        "median": "rent_median_city_type_bed",
        "mean": "rent_mean_city_type_bed",
        "count": "rent_count_city_type_bed",
    })
)

rent_group_city = (
    df_rentals.groupby(["city"], dropna=False)["monthly_rent"]
    .agg(["median", "mean", "count"])
    .reset_index()
    .rename(columns={
        "median": "rent_median_city",
        "mean": "rent_mean_city",
        "count": "rent_count_city",
    })
)

rent_group_province = (
    df_rentals.groupby(["province"], dropna=False)["monthly_rent"]
    .agg(["median", "mean", "count"])
    .reset_index()
    .rename(columns={
        "median": "rent_median_province",
        "mean": "rent_mean_province",
        "count": "rent_count_province",
    })
)

In [9]:
df_engine = df_sales.copy()

df_engine = df_engine.merge(
    rent_group_exact,
    on=["suburb", "property_type", "bedrooms"],
    how="left"
)

df_engine = df_engine.merge(
    rent_group_suburb_bed,
    on=["suburb", "bedrooms"],
    how="left"
)

df_engine = df_engine.merge(
    rent_group_suburb,
    on=["suburb"],
    how="left"
)

df_engine = df_engine.merge(
    rent_group_city_type_bed,
    on=["city", "property_type", "bedrooms"],
    how="left"
)

df_engine = df_engine.merge(
    rent_group_city,
    on=["city"],
    how="left"
)

df_engine = df_engine.merge(
    rent_group_province,
    on=["province"],
    how="left"
)

df_engine.shape

(9009, 44)

## 5. Estimate monthly rent for each sale listing

I create one final modeled rent field using the fallback hierarchy.

This gives me:
- `estimated_monthly_rent`
- `rent_estimation_source`

That way I know both the rent estimate and the level of confidence behind it.

In [10]:
rent_candidates = [
    "rent_median_suburb_type_bed",
    "rent_median_suburb_bed",
    "rent_median_suburb",
    "rent_median_city_type_bed",
    "rent_median_city",
    "rent_median_province",
]

source_labels = {
    "rent_median_suburb_type_bed": "suburb_property_type_bedrooms",
    "rent_median_suburb_bed": "suburb_bedrooms",
    "rent_median_suburb": "suburb",
    "rent_median_city_type_bed": "city_property_type_bedrooms",
    "rent_median_city": "city",
    "rent_median_province": "province",
}

df_engine["estimated_monthly_rent"] = df_engine[rent_candidates].bfill(axis=1).iloc[:, 0]

def pick_rent_source(row):
    for col in rent_candidates:
        if pd.notna(row[col]):
            return source_labels[col]
    return "unmatched"

df_engine["rent_estimation_source"] = df_engine.apply(pick_rent_source, axis=1)

df_engine[["listing_id", "suburb", "city", "property_type", "bedrooms", "estimated_monthly_rent", "rent_estimation_source"]].head()

,listing_id,suburb,city,property_type,bedrooms,estimated_monthly_rent,rent_estimation_source
0,T5332132,Broadacres,Sandton,Apartment,2.00,"10,700.00",suburb_property_type_bedrooms
1,T5405954,Sundowner,North Riding To Lanseria,Apartment,2.00,"18,200.00",suburb_bedrooms
2,T5374159,Jackal Creek Northriding,North Riding To Lanseria,Apartment,2.00,"6,650.00",suburb_property_type_bedrooms
3,T5078482,Braamfontein,Johannesburg Central,Apartment,2.00,"4,900.00",suburb_property_type_bedrooms
4,T4938016,Lonehill,Sandton,Apartment,2.00,"7,700.00",suburb_property_type_bedrooms


## 6. Define financial assumptions

Some financial values do not come directly from the scraped data, so I model them as explicit assumptions.

These are easy to change later for scenario testing.

In [11]:
ASSUMPTIONS = {
    "deposit_pct": 0.10,              # 10% deposit
    "annual_interest_rate": 0.11,     # 11% annual bond rate
    "loan_term_years": 20,            # 20-year bond
    "maintenance_rate": 0.08,         # 8% of monthly rent
    "vacancy_rate": 0.05,             # 5% vacancy
    "other_income": 0.00,             # extra monthly income default
    "transfer_cost_rate": 0.08,       # simple placeholder rate
    "property_value_growth_rate": 0.05
}

ASSUMPTIONS

{'deposit_pct': 0.1,
 'annual_interest_rate': 0.11,
 'loan_term_years': 20,
 'maintenance_rate': 0.08,
 'vacancy_rate': 0.05,
 'other_income': 0.0,
 'transfer_cost_rate': 0.08,
 'property_value_growth_rate': 0.05}

## 7. Core financial functions

I now implement the reusable functions that power the financial engine.


In [12]:
def safe_zero(value):
    return 0.0 if pd.isna(value) else float(value)

def amortized_bond_payment(loan_amount: float, annual_interest_rate: float, loan_term_years: float) -> float:
    loan_amount = safe_zero(loan_amount)
    annual_interest_rate = safe_zero(annual_interest_rate)
    loan_term_years = safe_zero(loan_term_years)

    if loan_amount <= 0 or loan_term_years <= 0:
        return 0.0

    monthly_rate = annual_interest_rate / 12
    total_payments = int(round(loan_term_years * 12))

    if monthly_rate == 0:
        return loan_amount / total_payments

    numerator = loan_amount * (monthly_rate * (1 + monthly_rate) ** total_payments)
    denominator = ((1 + monthly_rate) ** total_payments) - 1
    return numerator / denominator

def remaining_loan_balance(
    loan_amount: float,
    annual_interest_rate: float,
    loan_term_years: float,
    payments_made: int
) -> float:
    loan_amount = safe_zero(loan_amount)
    annual_interest_rate = safe_zero(annual_interest_rate)
    loan_term_years = safe_zero(loan_term_years)
    payments_made = int(max(0, safe_zero(payments_made)))

    if loan_amount <= 0 or loan_term_years <= 0:
        return 0.0

    monthly_rate = annual_interest_rate / 12
    total_payments = int(round(loan_term_years * 12))

    if payments_made >= total_payments:
        return 0.0

    if monthly_rate == 0:
        principal_paid = loan_amount * (payments_made / total_payments)
        return max(0.0, loan_amount - principal_paid)

    payment = amortized_bond_payment(loan_amount, annual_interest_rate, loan_term_years)

    balance = (
        loan_amount * (1 + monthly_rate) ** payments_made
        - payment * (((1 + monthly_rate) ** payments_made - 1) / monthly_rate)
    )
    return max(0.0, balance)

def classify_investment(cash_flow: float, dscr: float) -> str:
    if pd.isna(cash_flow) or pd.isna(dscr):
        return "Review"
    if cash_flow >= 0 and dscr > 1.20:
        return "Strong Investment"
    if cash_flow > -1000 and dscr >= 1.00:
        return "Moderate Investment"
    return "Weak Investment"

## 8. Compute the financial engine

### Purchase and financing
- purchase price = `purchase_price_Rands`
- deposit
- transfer costs
- loan amount
- bond payment

### Income and expenses
- estimated rent
- maintenance
- vacancy
- levies
- rates and taxes
- total expenses
- NOI

### Investment metrics
- cash flow
- DSCR
- rental yield
- ROI
- equity

In [14]:
# --- purchase price rule ---
df_engine["purchase_price"] = pd.to_numeric(df_engine["purchase_price"], errors="coerce")

# --- financing ---
df_engine["deposit_pct"] = ASSUMPTIONS["deposit_pct"]
df_engine["deposit_amount"] = df_engine["purchase_price"] * df_engine["deposit_pct"]

df_engine["transfer_cost_rate"] = ASSUMPTIONS["transfer_cost_rate"]
df_engine["transfer_costs"] = df_engine["purchase_price"] * df_engine["transfer_cost_rate"]

df_engine["total_cash_invested"] = df_engine["deposit_amount"] + df_engine["transfer_costs"]
df_engine["loan_amount"] = df_engine["purchase_price"] - df_engine["deposit_amount"]

df_engine["annual_interest_rate"] = ASSUMPTIONS["annual_interest_rate"]
df_engine["loan_term_years"] = ASSUMPTIONS["loan_term_years"]

df_engine["bond_payment"] = df_engine.apply(
    lambda row: amortized_bond_payment(
        loan_amount=row["loan_amount"],
        annual_interest_rate=row["annual_interest_rate"],
        loan_term_years=row["loan_term_years"],
    ),
    axis=1
)

# --- income ---
df_engine["other_income"] = ASSUMPTIONS["other_income"]
df_engine["gross_income"] = df_engine["estimated_monthly_rent"].fillna(0) + df_engine["other_income"]

# --- expenses ---
df_engine["levies_clean"] = df_engine["levies"].fillna(0)
df_engine["rates_taxes_clean"] = df_engine["rates_taxes"].fillna(0)

df_engine["maintenance_rate"] = ASSUMPTIONS["maintenance_rate"]
df_engine["vacancy_rate"] = ASSUMPTIONS["vacancy_rate"]

df_engine["maintenance_cost"] = df_engine["estimated_monthly_rent"].fillna(0) * df_engine["maintenance_rate"]
df_engine["vacancy_cost"] = df_engine["estimated_monthly_rent"].fillna(0) * df_engine["vacancy_rate"]

df_engine["total_expenses"] = (
    df_engine["levies_clean"]
    + df_engine["rates_taxes_clean"]
    + df_engine["maintenance_cost"]
    + df_engine["vacancy_cost"]
)

# --- noi and cash flow ---
df_engine["NOI"] = df_engine["gross_income"] - df_engine["total_expenses"]
df_engine["cash_flow"] = df_engine["NOI"] - df_engine["bond_payment"]

# --- risk / return metrics ---
df_engine["DSCR"] = np.where(
    df_engine["bond_payment"] > 0,
    df_engine["NOI"] / df_engine["bond_payment"],
    np.nan
)

df_engine["annual_rent"] = df_engine["estimated_monthly_rent"].fillna(0) * 12
df_engine["rental_yield"] = np.where(
    df_engine["purchase_price"] > 0,
    df_engine["annual_rent"] / df_engine["purchase_price"],
    np.nan
)

df_engine["annual_cash_flow"] = df_engine["cash_flow"] * 12
df_engine["ROI"] = np.where(
    df_engine["total_cash_invested"] > 0,
    df_engine["annual_cash_flow"] / df_engine["total_cash_invested"],
    np.nan
)

# --- simple equity snapshot after 12 payments ---
df_engine["payments_made_12m"] = 12
df_engine["remaining_balance_12m"] = df_engine.apply(
    lambda row: remaining_loan_balance(
        loan_amount=row["loan_amount"],
        annual_interest_rate=row["annual_interest_rate"],
        loan_term_years=row["loan_term_years"],
        payments_made=row["payments_made_12m"],
    ),
    axis=1
)

df_engine["equity_12m"] = df_engine["purchase_price"] - df_engine["remaining_balance_12m"]

# --- labels ---
df_engine["investment_label"] = df_engine.apply(
    lambda row: classify_investment(row["cash_flow"], row["DSCR"]),
    axis=1
)

## 9. Quick validation

I run a few sanity checks to make sure the engine behaves like a real investment model.

In [15]:
validation_report = {
    "sales_rows_preserved": len(df_engine) == len(df_sales),
    "listing_id_preserved": df_engine["listing_id"].equals(df_sales["listing_id"]),
    "critical_price_column_exists": "purchase_price" in df_engine.columns,
    "purchase_price_rands_non_null_pct": float(df_engine["purchase_price"].notna().mean()),
    "estimated_rent_non_null_pct": float(df_engine["estimated_monthly_rent"].notna().mean()),
    "bond_payment_non_negative": bool((df_engine["bond_payment"].fillna(0) >= 0).all()),
    "loan_amount_non_negative": bool((df_engine["loan_amount"].fillna(0) >= 0).all()),
}

validation_report

{'sales_rows_preserved': True,
 'listing_id_preserved': True,
 'critical_price_column_exists': True,
 'purchase_price_rands_non_null_pct': 1.0,
 'estimated_rent_non_null_pct': 1.0,
 'bond_payment_non_negative': True,
 'loan_amount_non_negative': True}

## 10. Review the new financial columns

I inspect the most important financial outputs.

In [16]:
financial_columns = [
    "listing_id",
    "title",
    "suburb",
    "city",
    "property_type",
    "bedrooms",
    "purchase_price",
    "estimated_monthly_rent",
    "rent_estimation_source",
    "deposit_amount",
    "transfer_costs",
    "total_cash_invested",
    "loan_amount",
    "bond_payment",
    "total_expenses",
    "NOI",
    "cash_flow",
    "DSCR",
    "rental_yield",
    "ROI",
    "remaining_balance_12m",
    "equity_12m",
    "investment_label",
]

df_engine[financial_columns].tail(20)

,listing_id,title,suburb,city,property_type,bedrooms,purchase_price,estimated_monthly_rent,rent_estimation_source,deposit_amount,transfer_costs,total_cash_invested,loan_amount,bond_payment,total_expenses,NOI,cash_flow,DSCR,rental_yield,ROI,remaining_balance_12m,equity_12m,investment_label
8989,SYNSALE08990,1 Bedroom Apartment in Sandton Central,Sandton Central,Sandton,Apartment,1.00,"2,617,000.00","13,600.00",suburb_property_type_bedrooms,"261,700.00","209,360.00","471,060.00","2,355,300.00","24,311.13","7,386.45","6,213.55","-18,097.59",0.26,0.06,-0.46,"2,320,951.92","296,048.08",Weak Investment
8990,SYNSALE08991,2 Bedroom House in Chartwell,Chartwell,North Riding To Lanseria,House,2.00,"1,724,000.00","19,200.00",suburb_property_type_bedrooms,"172,400.00","137,920.00","310,320.00","1,551,600.00","16,015.44","5,667.24","13,532.76","-2,482.67",0.84,0.13,-0.10,"1,528,972.53","195,027.47",Weak Investment
8991,SYNSALE08992,4 Bedroom House in North Riding,North Riding,North Riding To Lanseria,House,4.00,"1,990,000.00","27,550.00",suburb_property_type_bedrooms,"199,000.00","159,200.00","358,200.00","1,791,000.00","18,486.49","6,335.89","21,214.11","2,727.62",1.15,0.17,0.09,"1,764,881.28","225,118.72",Moderate Investment
8992,SYNSALE08993,2 Bedroom Apartment in Jackal Creek Northriding,Jackal Creek Northriding,North Riding To Lanseria,Apartment,2.00,"50,000.00","6,650.00",suburb_property_type_bedrooms,"5,000.00","4,000.00","9,000.00","45,000.00",464.48,"3,011.18","3,638.82","3,174.33",7.83,1.60,4.23,"44,343.75","5,656.25",Strong Investment
8993,SYNSALE08994,1 Bedroom Apartment in Ferndale,Ferndale,Randburg,Apartment,1.00,"50,000.00","6,400.00",suburb_property_type_bedrooms,"5,000.00","4,000.00","9,000.00","45,000.00",464.48,"3,360.05","3,039.95","2,575.46",6.54,1.54,3.43,"44,343.75","5,656.25",Strong Investment
8994,SYNSALE08995,1 Bedroom Apartment in Kensington,Kensington,Johannesburg Central,Apartment,1.00,"584,000.00","3,950.00",suburb_property_type_bedrooms,"58,400.00","46,720.00","105,120.00","525,600.00","5,425.18","3,016.46",933.54,"-4,491.65",0.17,0.08,-0.51,"517,935.01","66,064.99",Weak Investment
8995,SYNSALE08996,3 Bedroom House in Riverlea,Riverlea,Northcliff,House,3.00,"981,000.00","10,525.00",suburb_property_type_bedrooms,"98,100.00","78,480.00","176,580.00","882,900.00","9,113.19","3,927.52","6,597.48","-2,515.71",0.72,0.13,-0.17,"870,024.39","110,975.61",Weak Investment
8996,SYNSALE08997,2 Bedroom Apartment in Marshalltown,Marshalltown,Johannesburg Central,Apartment,2.00,"435,000.00","4,940.00",suburb_property_type_bedrooms,"43,500.00","34,800.00","78,300.00","391,500.00","4,041.02","2,609.42","2,330.58","-1,710.43",0.58,0.14,-0.26,"385,790.63","49,209.37",Weak Investment
8997,SYNSALE08998,4 Bedroom House in Petervale,Petervale,Sandton,House,4.00,"4,805,000.00","34,450.00",city_property_type_bedrooms,"480,500.00","384,400.00","864,900.00","4,324,500.00","44,636.99","8,417.22","26,032.78","-18,604.20",0.58,0.09,-0.26,"4,261,434.45","543,565.55",Weak Investment
8998,SYNSALE08999,2 Bedroom House in Erand Gardens,Erand Gardens,Midrand,House,2.00,"578,000.00","6,750.00",suburb_property_type_bedrooms,"57,800.00","46,240.00","104,040.00","520,200.00","5,369.44","2,717.34","4,032.66","-1,336.78",0.75,0.14,-0.15,"512,613.76","65,386.24",Weak Investment


## 11. Portfolio-level summary

This gives me a quick view of how the full dataset looks after running the financial engine.

In [17]:
summary = pd.DataFrame({
    "metric": [
        "sales_rows",
        "median_purchase_price",
        "median_estimated_monthly_rent",
        "median_bond_payment",
        "median_total_expenses",
        "median_NOI",
        "median_cash_flow",
        "median_DSCR",
        "median_rental_yield",
        "median_ROI",
    ],
    "value": [
        len(df_engine),
        df_engine["purchase_price"].median(),
        df_engine["estimated_monthly_rent"].median(),
        df_engine["bond_payment"].median(),
        df_engine["total_expenses"].median(),
        df_engine["NOI"].median(),
        df_engine["cash_flow"].median(),
        df_engine["DSCR"].median(),
        df_engine["rental_yield"].median(),
        df_engine["ROI"].median(),
    ]
})

summary

,metric,value
0,sales_rows,"9,009.00"
1,median_purchase_price,"1,148,000.00"
2,median_estimated_monthly_rent,"10,525.00"
3,median_bond_payment,"10,664.57"
4,median_total_expenses,"4,361.10"
5,median_NOI,"6,023.90"
6,median_cash_flow,"-3,988.62"
7,median_DSCR,0.59
8,median_rental_yield,0.11
9,median_ROI,-0.26


In [18]:
df_engine["investment_label"].value_counts(dropna=False)

investment_label
Weak Investment        6689
Strong Investment      1792
Moderate Investment     528
Name: count, dtype: int64

## 12. Keep the original sales columns together, then append financial fields

I want the sales dataset column names to flow properly.

So I preserve the original sales column order exactly as it appears in the attached sales file, and then I append the new financial-engine columns after that.

This makes the final export easier to understand and keeps it aligned with the source sales dataset.

In [19]:
original_sales_columns = df_sales.columns.tolist()

new_financial_columns = [
    col for col in df_engine.columns
    if col not in original_sales_columns
]

final_column_order = original_sales_columns + new_financial_columns

df_engine = df_engine[final_column_order]

print("First original columns:")
print(df_engine.columns[:len(original_sales_columns)].tolist())

print("\nFirst appended financial columns:")
print(new_financial_columns[:20])

First original columns:
['source_site', 'listing_id', 'listing_url', 'title', 'purchase_price', 'suburb', 'city', 'province', 'property_type', 'bedrooms', 'bathrooms', 'parking_spaces', 'garage', 'floor_area_sqm', 'land_area_sqm', 'levies', 'rates_taxes', 'description', 'listing_date', 'scraped_timestamp', 'pp_transaction_slug', 'pp_province_slug', 'pp_metro_slug', 'pp_city_slug', 'pp_suburb_slug', 'purchase_price_rands']

First appended financial columns:
['rent_median_suburb_type_bed', 'rent_mean_suburb_type_bed', 'rent_count_suburb_type_bed', 'rent_median_suburb_bed', 'rent_mean_suburb_bed', 'rent_count_suburb_bed', 'rent_median_suburb', 'rent_mean_suburb', 'rent_count_suburb', 'rent_median_city_type_bed', 'rent_mean_city_type_bed', 'rent_count_city_type_bed', 'rent_median_city', 'rent_mean_city', 'rent_count_city', 'rent_median_province', 'rent_mean_province', 'rent_count_province', 'estimated_monthly_rent', 'rent_estimation_source']


## 13. Export the enriched financial dataset

The output is the original expanded sales dataset plus the financial-engine fields.

This becomes the core dataset for:
- feature engineering
-

In [20]:
output_path = Path("data/interim/financial_engine_sales_dataset.csv")
df_engine.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Output shape: {df_engine.shape}")

Saved to: data\interim\financial_engine_sales_dataset.csv
Output shape: (9009, 75)
